# 2.1 Bluesky — Exploratory Data Analysis

Startpunt: **Bronze layer** (`bsky_US_2024_raw.csv`).  
We leiden de candidate-cluster kolom zelf af uit de `query`-hashtag — zelfde indeling als `data_retrieval/2_bluesky.ipynb`.  
Downstream notebooks: network (2.2) · textual (2.3) · NLP (2.4) · sentiment (2.5).

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.path.abspath("."), "..", ".."))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
from house_style import *

apply_style()

BRONZE = "../../Data/1_Bronze/Bluesky/bsky_US_2024_raw.csv"
df = pd.read_csv(BRONZE, parse_dates=["timestamp"])
print(f"Loaded {len(df):,} rows x {df.shape[1]} columns")
df.head(3)

Loaded 32,862 rows x 13 columns


,uri,author,display,text,timestamp,likes,reposts,replies,mentions,is_reply,post_type,query,parent_uri
0,at://did:plc:taylv7omre5pg7arsuidtwg4/app.bsky...,outrowes.bsky.social,Wes,Os Swing States são os estados americanos onde...,2024-11-04T23:53:52.367Z,1,1,0,[],False,post,#USElection2024,NaN
1,at://did:plc:lpqouffbl3gbah3r23vh7odp/app.bsky...,runhudi.bsky.social,Yehuda M.,Canadian election prediction: Americans will g...,2024-11-04T23:44:15.319Z,0,0,1,[],False,post,#USElection2024,NaN
2,at://did:plc:3ixumkojw3dpsfhdyjodfxmk/app.bsky...,manabouttown.bsky.social,NaN,Why are swinger states so important in the #US...,2024-11-04T23:35:21.961Z,0,0,0,[],False,post,#USElection2024,NaN


## Candidate cluster afleiding

Hashtags gegroepeerd in drie clusters op basis van de retrieval-definitie.

In [ ]:
TRUMP_TAGS = {
    "#Trump2024", "#TrumpVance", "#VoteTrump", "#Trump", "#DonaldTrump",
    "#MAGA", "#MAGA2024", "#AmericaFirst", "#TrumpRally",
    "#Republicans", "#GOP", "#Republican",
    "#JDVance", "#Vance2024", "#VanceVP",
    "#RNC2024", "#RepublicanConvention", "#Project2025", "#TrumpDebate",
}

HARRIS_TAGS = {
    "#Harris2024", "#KamalaHarris2024", "#KamalaHarris",
    "#HarrisWalz", "#VoteHarris", "#VoteBlue", "#VoteKamala",
    "#Kamala2024", "#Kamala", "#Harris",
    "#TimWalz", "#Walz2024", "#WalzVP",
    "#DNC2024", "#DemConvention", "#DemocraticConvention",
    "#WeAreNotGoingBack", "#WinWithKamala",
    "#Democrats", "#Democrat",
    "#BidenDropsOut", "#BidenOut", "#BidenWithdraws",
}

def map_candidate(query):
    q = str(query).strip()
    if q in TRUMP_TAGS:  return "CandidateA"  # Trump cluster
    if q in HARRIS_TAGS: return "CandidateB"  # Harris cluster
    return "Neutral"

df["candidate"] = df["query"].apply(map_candidate)
df["date"]      = df["timestamp"].dt.normalize()
df["candidate"].value_counts()

## Base table

In [ ]:
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

## Null check

In [ ]:
null_pct = (df.isnull().sum() / len(df) * 100).round(1)
null_pct[null_pct > 0].rename("null %").to_frame()

## Key counts per candidate cluster

In [ ]:
summary = pd.DataFrame({
    "total_posts"   : df.groupby("candidate").size(),
    "unique_authors": df.groupby("candidate")["author"].nunique(),
    "avg_likes"     : df.groupby("candidate")["likes"].mean().round(2),
    "avg_reposts"   : df.groupby("candidate")["reposts"].mean().round(2),
    "pct_reply"     : (df.groupby("candidate")["is_reply"].mean() * 100).round(1),
})
summary

## Dagelijks postvolume

In [ ]:
COLORS = {"CandidateA": REPUBLICAN, "CandidateB": DEMOCRAT, "Neutral": NEUTRAL}

daily = (
    df.groupby(["date", "candidate"])
      .size()
      .reset_index(name="posts")
)

fig, ax = plt.subplots(figsize=(13, 4))
for cand, grp in daily.groupby("candidate"):
    ax.plot(grp["date"], grp["posts"], label=cand,
            color=COLORS.get(cand, NEUTRAL), linewidth=1.4)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.set_ylabel("Posts per dag", color=TEXT_MUTED)
ax.set_title("Dagelijks Bluesky postvolume per candidate cluster",
             color=TEXT_PRIMARY, pad=10)
ax.legend(frameon=False, labelcolor=TEXT_PRIMARY)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## Like distributie (log schaal)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for cand, grp in df.groupby("candidate"):
    ax.hist(np.log1p(grp["likes"]), bins=40, alpha=0.55,
            label=cand, color=COLORS.get(cand, NEUTRAL))
ax.set_xlabel("log(1 + likes)", color=TEXT_MUTED)
ax.set_ylabel("Count", color=TEXT_MUTED)
ax.set_title("Like distributie per candidate cluster", color=TEXT_PRIMARY, pad=10)
ax.legend(frameon=False, labelcolor=TEXT_PRIMARY)
plt.tight_layout()
plt.show()

## Top-10 hashtags (query)

In [ ]:
top_queries = df["query"].value_counts().head(10)

fig, ax = plt.subplots(figsize=(8, 4))
bar_colors = [COLORS.get(map_candidate(q), NEUTRAL) for q in top_queries.index]
ax.barh(top_queries.index[::-1], top_queries.values[::-1], color=bar_colors[::-1])
ax.set_xlabel("Aantal posts", color=TEXT_MUTED)
ax.set_title("Top-10 queries (hashtags)", color=TEXT_PRIMARY, pad=10)
plt.tight_layout()
plt.show()